## Select Event based on OLR dataset

### Import package

In [1]:
import numpy as np
import pandas as pd
import netCDF4 as nc

from tqdm import tqdm
from datetime import datetime, timedelta
from itertools import product
from typing import Tuple

from matplotlib import pyplot as plt


### Helper function

In [2]:
def generate_1d_tukey(x: np.ndarray, center: float, flat_width: float, taper_width: float) -> np.ndarray:
    """Evaluates a continuous 1D Tukey window over an array x."""
    dist = np.abs(x - center)
    half_flat = flat_width / 2.0
    
    # Initialize the window
    w = np.zeros_like(x)
    
    # 1. Flat top region
    w[dist <= half_flat] = 1.0
    
    # 2. Tapered region
    taper_mask = (dist > half_flat) & (dist <= half_flat + taper_width)
    w[taper_mask] = 0.5 * (1 + np.cos(np.pi * (dist[taper_mask] - half_flat) / taper_width))
    
    return w

### Load data

In [3]:
# Setup time range
time_range = pd.date_range(start="2006-01-01", end="2017-12-31", freq="D")

# Load OLR data
with nc.Dataset("/work/DATA/Satellite/OLR/olr_anomaly.nc") as olr_ds:
    ## set starting time stamp
    starting_time: datetime = datetime.strptime(olr_ds["time"].units.split("since ")[1][:-2], "%Y-%m-%d %H:%M:%S")

    ## Convert time variable to datetime objects
    time: np.ndarray = np.array([starting_time + timedelta(hours=int(t)) for t in olr_ds["time"][:]])

    ## set range of time
    time_mask: np.ndarray = (time >= time_range[0]) & (time <= time_range[-1])

    ## set range for latitude
    lat_mask: np.ndarray = (olr_ds["lat"][:] >= -5) & (olr_ds["lat"][:] <= 5)

    ## Load OLR anomaly data and apply masks
    lat: np.ndarray = olr_ds["lat"][lat_mask]
    olr_anom: np.ndarray = olr_ds["olr"][time_mask, lat_mask, :]

### Preprocessing

In [4]:
# symmetrizing data
olr_symm: np.ndarray = (olr_anom + np.flip(olr_anom, axis=1)) / 2

# average over latitude with area weighting (cosine of latitude)
olr_avg: np.ndarray = np.nanmean(olr_symm * np.cos(np.radians(lat)[None, :, None]), axis=1)

ntime, nlon = olr_avg.shape

### Apply bandpass filter

In [5]:
# generate wavenumber and frequency arrays
wnum: np.ndarray = np.fft.fftfreq(nlon, d=1/nlon)
freq: np.ndarray = np.fft.fftfreq(ntime, d=1)

wnums, freqs = np.meshgrid(wnum, freq)

dk: float = wnum[1] - wnum[0]
df: float = freq[1] - freq[0]

# targeting wavenumber and frequency ranges
wnum_target: np.ndarray = np.arange(-15, 15+2*dk, 2*dk)
freq_target: np.ndarray = np.arange(np.abs(freq).min()+10*df, np.abs(freq).max()-10*df, 10*df)

# perform 2D FFT to assigned grid
olr_fft: np.ndarray = np.fft.ifft(np.fft.fft(olr_avg, axis=1) / nlon, axis=0)

# design different targeting windows
target_grids = list(product(wnum_target, freq_target))

# Loop over targeting wavenumber and frequency band
for target_grid in tqdm(target_grids):

    k0, f0 = target_grid # defining the center of this window

    # Generater Tukey taper
    ## Design for the Tukey taper
    k_flat, k_taper = 2*dk, 4*dk
    f_flat, f_taper = 3*df, 5*df

    ## Taper
    W_k_pos: np.ndarray = generate_1d_tukey(wnums, k0, k_flat, k_taper)
    W_f_pos: np.ndarray = generate_1d_tukey(freqs, f0, f_flat, f_taper)

    tukey_taper_pos: np.ndarray = W_k_pos * W_f_pos

    W_k_neg: np.ndarray = generate_1d_tukey(wnums, -k0, k_flat, k_taper)
    W_f_neg: np.ndarray = generate_1d_tukey(freqs, -f0, f_flat, f_taper)
    
    tukey_taper_neg: np.ndarray = W_k_neg * W_f_neg

    tukey_taper: np.ndarray = tukey_taper_pos + tukey_taper_neg

    ## mask out other band
    olr_fft_filtered: np.ndarray = olr_fft.copy() * tukey_taper

    ## inverse transform
    olr_filtered: np.ndarray = np.real(np.fft.fft(np.fft.ifft(olr_fft_filtered, axis=1), axis=0))

    ## filter lower peak value 
    ### find threshold
    threshold: float = float(np.nanmean(olr_filtered) - 3.32*np.nanstd(olr_filtered))

    ### find the location lower than the threshold
    sig_loc = np.where(olr_filtered <= threshold)

    # save index file
    np.savetxt(f"/home/b11209013/KW_CloudSat/Files/ERA5_GRIB/significant_loc/{target_grid}.txt", np.column_stack(sig_loc), fmt="%d")


100%|██████████| 3488/3488 [07:48<00:00,  7.44it/s]
